# Phase 2 - San Semantic Retrieval Research

Purpose:
- Build the missing Phase 2 research bridge between Hoang Phase 1 routing and Hoang Phase 3 grounded QA.
- Inspect how the aviation corpus is chunked, embedded, indexed, searched, scored, and passed to the LLM layer.
- Compare retrieval strategies: `bm25`, `semantic`, `hybrid`, `metadata_first`, and `hybrid_rrf`.

This notebook mirrors the app/runtime implementation in `aviation_rag/phase2_indexing.py` and `aviation_rag/phase2_retrieval_engine.py`.

## 1. Setup

The default architecture wants `sentence-transformers/all-MiniLM-L6-v2` for 384-dim dense embeddings. If the model is not cached or the machine cannot access HuggingFace, the runtime falls back explicitly to `tfidf_svd_faiss_fallback` and reports that in diagnostics.

For fast local research, set `USE_FAST_FALLBACK = True`. For final dense retrieval validation, set it to `False` and ensure the MiniLM model is available.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("LANGSMITH_TRACING", "false")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "false")
os.environ.setdefault("RETRIEVAL_MAX_DOCS", "500")
os.environ.setdefault("PHASE2_INDEX_DIR", str(PROJECT_ROOT / "artifacts" / "phase2_index_notebook"))

USE_FAST_FALLBACK = True
if USE_FAST_FALLBACK:
    os.environ["PHASE2_EMBEDDING_MODEL"] = "tfidf_svd_fallback"

from aviation_rag.config import Settings, ensure_artifact_dirs
from aviation_rag.phase1_hoang_intent_routing import Phase1HoangIntentRouting
from aviation_rag.phase2_indexing import build_corpus_chunks
from aviation_rag.phase2_san_faiss_retrieval import Phase2SanFaissRetrieval
from aviation_rag.phase2_metrics import evaluate_ranking
from aviation_rag.io_utils import write_jsonl

settings = Settings()
ensure_artifact_dirs(settings)
settings

## 2. Corpus Chunking Inspection

Phase 2 starts from the shared aviation corpus. The runtime extracts text fields, normalizes aviation abbreviations, deduplicates repeated chunks, and attaches metadata such as airport, weather, component, problem type, and inferred document type.

In [ ]:
chunks = build_corpus_chunks(settings)
print("chunk_count:", len(chunks))
print("first_chunk:")
print(json.dumps(chunks[0].__dict__, ensure_ascii=False, indent=2)[:1800])

## 3. Phase 1 Input Contract

Phase 1 sends Phase 2 a normalized object containing the query, predicted intent, expanded queries, rewritten query, and retrieval plan. Phase 2 does not guess user intent again; it respects this routing contract.

In [ ]:
phase1 = Phase1HoangIntentRouting(settings)
query = "engine warning checklist after takeoff"
phase1_output = phase1.build_output(query, top_k=5)
print(phase1_output.model_dump_json(indent=2))

## 4. Build Phase 2 Retrieval Engine

The retrieval engine supports:
- Dense semantic retrieval via MiniLM embeddings + L2 normalization + FAISS `IndexFlatIP`.
- BM25 lexical retrieval for exact technical/procedure terms.
- Metadata scoring/filtering for operating conditions and structured fields.
- Weighted hybrid fusion and Reciprocal Rank Fusion (`hybrid_rrf`).

In [ ]:
retrieval = Phase2SanFaissRetrieval(settings)
print("available:", retrieval.available)
print("build_error:", retrieval.build_error)
print("index_info:")
print(json.dumps(retrieval.info.__dict__ if retrieval.info else {}, ensure_ascii=False, indent=2))

## 5. Run One Retrieval Strategy

Inspect returned documents, score decomposition, and diagnostics. These diagnostics are what the Streamlit UI uses for the ?How do I know?? panel.

In [ ]:
phase1_output = phase1.build_output(query, top_k=5, strategy="hybrid")
phase2_output = retrieval.retrieve(phase1_output)

print("diagnostics:")
print(json.dumps(phase2_output.retrieval_diagnostics, ensure_ascii=False, indent=2))

print("top docs:")
for rank, doc in enumerate(phase2_output.topk_docs, start=1):
    print(rank, doc.doc_id, doc.metadata.get("document_type"), doc.scores)
    print(doc.chunk_text[:350].replace("\n", " "))
    print("---")

## 6. Compare Methods

A good demo should not hide the algorithm choice. This cell compares the same query under each strategy so we can see which retrieval behavior is better for a given intent.

In [ ]:
strategies = ["bm25", "semantic", "hybrid", "metadata_first", "hybrid_rrf"]
comparison_rows = []
for strategy in strategies:
    row = phase1.build_output(query, top_k=5, strategy=strategy)
    output = retrieval.retrieve(row)
    first_doc = output.topk_docs[0] if output.topk_docs else None
    comparison_rows.append({
        "strategy": strategy,
        "top_doc": first_doc.doc_id if first_doc else None,
        "top_type": first_doc.metadata.get("document_type") if first_doc else None,
        "top_final": first_doc.scores.get("final") if first_doc else None,
        "fusion": output.retrieval_diagnostics.get("fusion_method"),
        "latency_ms": output.retrieval_diagnostics.get("latency_ms"),
        "embedding_backend": output.retrieval_diagnostics.get("embedding_backend"),
    })

comparison_rows

## 7. Lightweight Retrieval Evaluation

This is not a full academic benchmark yet, but it gives local sanity checks for Precision@K, Recall@K, MRR, and latency. In a stronger evaluation, replace heuristic `expected_terms` with labeled relevant document IDs.

In [ ]:
benchmark_cases = [
    {"query": "engine warning checklist after takeoff", "strategy": "bm25", "expected_terms": ["engine", "warning", "checklist"]},
    {"query": "crosswind turbulence during final approach", "strategy": "metadata_first", "expected_terms": ["crosswind", "turbulence", "approach"]},
    {"query": "engine failure after takeoff with emergency return", "strategy": "semantic", "expected_terms": ["engine", "takeoff", "emergency"]},
]

def doc_is_relevant(doc, terms):
    text = " ".join([doc.chunk_text, " ".join(str(v) for v in doc.metadata.values())]).lower()
    return any(term.lower() in text for term in terms)

metric_rows = []
for case in benchmark_cases:
    row = phase1.build_output(case["query"], top_k=5, strategy=case["strategy"])
    output = retrieval.retrieve(row)
    retrieved_ids = [doc.doc_id for doc in output.topk_docs]
    relevant_ids = [doc.doc_id for doc in output.topk_docs if doc_is_relevant(doc, case["expected_terms"])]
    metric = evaluate_ranking(retrieved_ids, relevant_ids, k=5, latency_ms=output.retrieval_diagnostics.get("latency_ms", 0.0))
    metric_rows.append({"query": case["query"], "strategy": case["strategy"], **metric.__dict__})

metric_rows

## 8. Export Phase 2 Artifact

The artifact is the handoff from San Phase 2 to Hoang Phase 3. It contains `query_id`, `predicted_intent`, `topk_docs`, and `retrieval_diagnostics`.

In [ ]:
artifact_path = settings.artifacts_dir / "phase2_san_retrieval_output.jsonl"
write_jsonl(artifact_path, [phase2_output])
print("wrote:", artifact_path)
print(artifact_path.read_text(encoding="utf-8")[:1200])

## 9. Research Conclusion

Phase 2 is the evidence engine. It receives intent-aware routing from Phase 1 and returns ranked, scored, citation-ready context for Phase 3.

Recommended interpretation:
- `bm25`: best when exact checklist/component terms matter.
- `semantic`: best when the user describes an incident narratively.
- `metadata_first`: best when weather, airport, runway, operating condition, or structured fields matter.
- `hybrid`: robust default combining semantic + keyword + metadata.
- `hybrid_rrf`: rank-fusion alternative that is often stable when score scales differ.